In [ ]:
OPENDS_NAME             = "MO"
TASK                    = "join"

# GT_PATH               = f"{TASK}/ground truth/opendata_{TASK}_ground_truth.csv"
GT_PATH                 = 'join_tables.csv'

#QUERY_TABLES_PATH      = f"{TASK}/opendata_{TASK}_query/tables"
QUERY_TABLES_PATH       = 'join_tables.csv'

# DATASET_PATH          = f"datasets_{OPENDS_NAME}"
DATASET_PATH            = f"datasets_{OPENDS_NAME}/comune di modena/"

MAX_UNIQUE_VALUES       = 10
N_REPRESENTATIVE_ROWS   = 3

In [2]:
import os
os.environ['OLLAMA_HOST'] = 'http://localhost:11434'
os.environ['no_proxy'] = 'localhost,127.0.0.1'

In [3]:
import pandas as pd
ground_truth = pd.read_csv(GT_PATH)
# select only query_table and candidate_table that starts for UK
#uk_tables = ground_truth[(ground_truth['query_table'].str.startswith(OPENDS_NAME)) & (ground_truth['candidate_table'].str.startswith(OPENDS_NAME))]
#uk_tables = uk_tables.reset_index(drop=True)
uk_tables = ground_truth
uk_tables

,query_table,candidate_table
0,Numero di immigrati stranieri a Modena.csv,Numero di immigrati stranieri a Modena per cla...
1,Popolazione residente a Modena dal 1995.csv,Serie storiche della popolazione residente.csv
2,Famiglie aggregate per numero di componenti la...,Residenti nelle singole sezioni di censimento ...
3,Indirizzario delle sezioni elettorali.csv,Sezioni elettorali.csv
4,Nomi di battesimo piu frequenti di sesso masch...,Popolazione residente a Modena dal 1995.csv
5,Toponomastica stradale.csv,Numerazione civica del territorio comunale.csv
6,Stradario comunale.csv,Bacini scolastici.csv
7,Esercizi commerciali.csv,Ristoranti.csv
8,Distributori.csv,Strutture ricettive.csv


In [4]:
#uk_tables = uk_tables[~uk_tables["query_table"].str.contains("__") & ~uk_tables["candidate_table"].str.contains("__")]
#uk_tables

In [5]:
uk_tables = uk_tables.reset_index()

In [6]:
import os
deepseek = {
    "config_list": [
        {
        "model": "llama-3.1-8b-instant",
        "api_key": os.environ.get("GROQ_API_KEY") ,
        "api_type": "groq",
        "max_tokens": 8192,
        "frequency_penalty": 0.5,
        "presence_penalty": 0.2,
        "temperature": 0
        }
    ],
    "cache_seed": None
}

mistral = {
    "config_list": [
        {
        "model": "llama-3.1-8b-instant",
        "api_key": os.environ.get("GROQ_API_KEY") ,
        "api_type": "groq",
        "max_tokens": 8192,
        "frequency_penalty": 0.5,
        "presence_penalty": 0.2,
        "temperature": 0
        }
    ],
    "cache_seed": None
}

In [7]:
mistral_sql = {
    "config_list": [
       {
            "model": "llama3_custom_8",  # Loaded with LiteLLM command
            "api_key": "NotRequired",  # Not needed
            "base_url": "http://127.0.0.1:8090",  # Your LiteLLM URL
            "price": [0, 0],  # Put in price per 1K tokens [prompt, response] as free!
            "timeout": 1200
       }
    ],
    "cache_seed": None,  # Disable caching.
}

qwen_small = {
    "config_list": [
        {
            "model": "llama3_custom_8",  # Loaded with LiteLLM command
            "api_key": "NotRequired",  # Not needed
            "base_url": "http://127.0.0.1:8090",  # Your LiteLLM URL
            "price": [0, 0],  # Put in price per 1K tokens [prompt, response] as free!
             "timeout": 1200
       }
    ],
    "cache_seed": None,  # Disable caching.
}

In [8]:
from autogen.coding import LocalCommandLineCodeExecutor
executor = LocalCommandLineCodeExecutor(
    timeout=10,  # Timeout for each code execution in seconds.
    work_dir='./coding',  # Use the temporary directory to store the code files.
)

flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.


In [9]:
from autogen import AssistantAgent
assistant = AssistantAgent(
    name="Assistant",
    system_message="""
        You are an SQL expert of joinable tables.
        You will take two joinable tables from a datalake, each containing five example entries for each column. 
        You are required to GENERATE a natural language question that necessitate joining these two tables. 
        Do not generate the answer.
        Report also the corresponding SQL code block.
        Be sure to use the correct column names in the SQL block, and put column names inside ``.
        Generate the natural language question do not add any other info.
        Make sure that SQL is write in the way that always return at least one result.
        Make sure that table names are inside ''.
    """,
    llm_config=mistral_sql,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
)


In [10]:
import os
table1_name = uk_tables[['query_table']].iloc[3].values[0]
table2_name = uk_tables[["candidate_table"]].iloc[3].values[0]

uk_tables_csv = os.listdir(DATASET_PATH)
#joinable_tables_csv = os.listdir(QUERY_TABLES_PATH)

# Initialize file_path variables
file_path_table1 = None
file_path_table2 = None

# Function to get the file path for a given table name
def get_file_path(table_name):
    if table_name in uk_tables_csv:
        return os.path.join(DATASET_PATH, table_name)
    elif table_name in joinable_tables_csv:
        return os.path.join(QUERY_TABLES_PATH, table_name)
    return None

# Get file paths for table1_name and table2_name
file_path_table1 = get_file_path(f'{table1_name}')
file_path_table2 = get_file_path(f'{table2_name}')

# Output results
if file_path_table1:
    print(f"File path for table1: {file_path_table1}")
else:
    print("Table1 not found in either dataset.")

if file_path_table2:
    print(f"File path for table2: {file_path_table2}")
else:
    print("Table2 not found in either dataset.")

File path for table1: datasets_MO/comune di modena/Indirizzario delle sezioni elettorali.csv
File path for table2: datasets_MO/comune di modena/Sezioni elettorali.csv


In [11]:
from autogen import UserProxyAgent
user_proxy = UserProxyAgent(
    name="User_proxy",
    system_message="A human admin. Execute code when needed",
    code_execution_config={
        "last_n_messages": 3,
        "executor": LocalCommandLineCodeExecutor(work_dir="groupchat"),
    },
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    max_consecutive_auto_reply=5
)

In [12]:
import json
from typing import Dict
from autogen import AssistantAgent

def check_termination(msg: Dict):
    if "tool_responses" not in msg:
        return False
    json_str = msg["tool_responses"][0]["content"]
    obj = json.loads(json_str)
    return "error" not in obj or obj["error"] is None and obj["reward"] == 1


sql_writer = AssistantAgent(
    "sql_writer",
    system_message="You are good at writing SQL queries. Always respond with a function call to execute_sql().",
    is_termination_msg=check_termination,
    llm_config=mistral_sql,
)

In [13]:
from typing import Annotated, Dict
import sqlite3
import pandas as pd
import csv


@sql_writer.register_for_llm(description="Function for executing SQL query and returning a response")
@user_proxy.register_for_execution()
def execute_sql(
    question:  Annotated[str, "Think about the corresponding natural language question to SQL."],
    human_like_question: Annotated[str, "Convert question without using technical or specific column names. Instead, replace them with more relatable and descriptive terms to make the question feel intuitive and conversational."],
    reflection: Annotated[str, "Think about what to do"], sql: Annotated[str, "SQL query"], table1: Annotated[str, "name of the first table"], table2: Annotated[str, "name of the second table"]
) -> Annotated[Dict[str, str], "Dictionary with keys 'result' and 'error'"]:
        connection = None
        error = None
        result = None
        
        try:
            # Connect to the SQLite database
            connection = sqlite3.connect('MO_tables.db')
            cursor = connection.cursor()
            
            # Execute the SQL query
            cursor.execute(sql)
            result = cursor.fetchall()
            
            # Commit changes if the query modifies the database
            connection.commit()
            
        except sqlite3.Error as e:
            # Handle SQL errors
            error = str(e)
        
        finally:
            if connection:
                # Close the connection
                connection.close()
        
        if error:
            return {
                "error": error,
                "wrong_result": result if result else "No result",
                "correct_result": "N/A"
            }
        
        
        elif len(result) == 0:
            return {
                "error":  'Empty result',
                "result": "No result",
                "correct_result": "N/A"
            }
            
            
        else:
            # save results to csv
            # Check if the CSV file exists
            file_exists = os.path.isfile('query_results.csv')
 
            with open('query_results.csv', 'a', newline='') as csvfile:
                csv_writer = csv.writer(csvfile)
                if not file_exists:
                    csv_writer.writerow(['question', 'query', 'sql', 'table1', 'table2'])  # Write header
            
                csv_writer.writerow([human_like_question, question, sql, table1, table2])

            return {
                "result": result,
            }


In [14]:
sql_rewriter = AssistantAgent(
    "rewriter",
    system_message="Given an SQL query and an error try to modify the sql to avoid the error. Write only the new sql.",
    is_termination_msg=check_termination,
    llm_config=mistral_sql,
)

In [15]:
from autogen import GroupChat, GroupChatManager

groupchat = GroupChat(agents=[sql_writer, user_proxy, sql_rewriter], messages=[], max_round=30, speaker_selection_method="round_robin")
manager = GroupChatManager(groupchat=groupchat, llm_config=mistral_sql, system_message="When the task is completed say TERMINATE",)

In [16]:
# import os

# table1_name = uk_tables[['query_table']].iloc[4491].values[0]
# table2_name = uk_tables[["candidate_table"]].iloc[4491].values[0]

# table_list = os.listdir('./representative_table_rows')

# with open(f'./representative_table_rows/{table1_name}.txt', 'r') as file:
#     table1 = file.read()
    
# with open(f'./representative_table_rows/{table2_name}.txt', 'r') as file:
#     table2 = file.read()

In [17]:
tables = list(uk_tables['query_table'].values) + list(uk_tables['candidate_table'].values)

In [18]:
tables

['Numero di immigrati stranieri a Modena.csv',
 'Popolazione residente a Modena dal 1995.csv',
 'Famiglie aggregate per numero di componenti la famiglia.csv',
 'Indirizzario delle sezioni elettorali.csv',
 'Nomi di battesimo piu frequenti di sesso maschile nel 2021.csv',
 'Toponomastica stradale.csv',
 'Stradario comunale.csv',
 'Esercizi commerciali.csv',
 'Distributori.csv',
 'Numero di immigrati stranieri a Modena per classi di eta.csv',
 'Serie storiche della popolazione residente.csv',
 'Residenti nelle singole sezioni di censimento del 2011: dal 2012 al 2021.csv',
 'Sezioni elettorali.csv',
 'Popolazione residente a Modena dal 1995.csv',
 'Numerazione civica del territorio comunale.csv',
 'Bacini scolastici.csv',
 'Ristoranti.csv',
 'Strutture ricettive.csv']

In [19]:
# Function to check if a file exists in either path
def file_exists(file_name):
    return os.path.exists(os.path.join('representative_table_rows_MO', f'{file_name[:-3]}.txt')) or os.path.exists(os.path.join(QUERY_TABLES_PATH, file_name))

# Filter the list to include only existing files
#filtered_tables = list(filter(file_exists, tables))

filtered_tables = tables

In [20]:
filtered_tables

['Numero di immigrati stranieri a Modena.csv',
 'Popolazione residente a Modena dal 1995.csv',
 'Famiglie aggregate per numero di componenti la famiglia.csv',
 'Indirizzario delle sezioni elettorali.csv',
 'Nomi di battesimo piu frequenti di sesso maschile nel 2021.csv',
 'Toponomastica stradale.csv',
 'Stradario comunale.csv',
 'Esercizi commerciali.csv',
 'Distributori.csv',
 'Numero di immigrati stranieri a Modena per classi di eta.csv',
 'Serie storiche della popolazione residente.csv',
 'Residenti nelle singole sezioni di censimento del 2011: dal 2012 al 2021.csv',
 'Sezioni elettorali.csv',
 'Popolazione residente a Modena dal 1995.csv',
 'Numerazione civica del territorio comunale.csv',
 'Bacini scolastici.csv',
 'Ristoranti.csv',
 'Strutture ricettive.csv']

In [21]:
uk_tables = uk_tables[(uk_tables['query_table'].isin(filtered_tables)) & (uk_tables['candidate_table'].isin(filtered_tables))]

In [22]:
uk_tables

,index,query_table,candidate_table
0,0,Numero di immigrati stranieri a Modena.csv,Numero di immigrati stranieri a Modena per cla...
1,1,Popolazione residente a Modena dal 1995.csv,Serie storiche della popolazione residente.csv
2,2,Famiglie aggregate per numero di componenti la...,Residenti nelle singole sezioni di censimento ...
3,3,Indirizzario delle sezioni elettorali.csv,Sezioni elettorali.csv
4,4,Nomi di battesimo piu frequenti di sesso masch...,Popolazione residente a Modena dal 1995.csv
5,5,Toponomastica stradale.csv,Numerazione civica del territorio comunale.csv
6,6,Stradario comunale.csv,Bacini scolastici.csv
7,7,Esercizi commerciali.csv,Ristoranti.csv
8,8,Distributori.csv,Strutture ricettive.csv


In [23]:
# remove .csv from query_table and candidate_table
uk_tables['query_table'] = uk_tables['query_table'].str.replace('.csv', '')
uk_tables['candidate_table'] = uk_tables['candidate_table'].str.replace('.csv', '')
uk_tables

,index,query_table,candidate_table
0,0,Numero di immigrati stranieri a Modena,Numero di immigrati stranieri a Modena per cla...
1,1,Popolazione residente a Modena dal 1995,Serie storiche della popolazione residente
2,2,Famiglie aggregate per numero di componenti la...,Residenti nelle singole sezioni di censimento ...
3,3,Indirizzario delle sezioni elettorali,Sezioni elettorali
4,4,Nomi di battesimo piu frequenti di sesso masch...,Popolazione residente a Modena dal 1995
5,5,Toponomastica stradale,Numerazione civica del territorio comunale
6,6,Stradario comunale,Bacini scolastici
7,7,Esercizi commerciali,Ristoranti
8,8,Distributori,Strutture ricettive


In [24]:
table1_name = uk_tables[['query_table']].iloc[0].values[0]
table2_name = uk_tables[["candidate_table"]].iloc[0].values[0]
table_list = os.listdir('./representative_table_rows_MO')

with open(f'./representative_table_rows_MO/{table1_name}.txt', 'r') as file:
    table1 = file.read()
    
with open(f'./representative_table_rows_MO/{table2_name}.txt', 'r') as file:
    table2 = file.read()
    
conn = sqlite3.connect('MO_tables.db')
cursor = conn.cursor()
cursor.execute(f'PRAGMA table_info("{table2}");')

# Fetch all results from the executed query
schema = cursor.fetchall()

# Close the connection
conn.close()

In [25]:
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.schema import CreateTable
engine = create_engine('sqlite:///MO_tables.db')
metadata = MetaData()

# Reflect the table
print(str(CreateTable(Table(table1_name, metadata, autoload_with=engine)).compile(engine)))


CREATE TABLE "Numero di immigrati stranieri a Modena" (
	"FID" TEXT, 
	"ANNO" INTEGER, 
	"CITTADINANZA" TEXT, 
	"AREA" TEXT, 
	"PROVENIENZA" TEXT, 
	"SESSO" TEXT, 
	"IMMIGRATI" INTEGER, 
	"OBJECTID" INTEGER, 
	"SHAPE" TEXT
)




In [26]:
uk_tables.drop_duplicates(subset=['query_table', 'candidate_table'])

,index,query_table,candidate_table
0,0,Numero di immigrati stranieri a Modena,Numero di immigrati stranieri a Modena per cla...
1,1,Popolazione residente a Modena dal 1995,Serie storiche della popolazione residente
2,2,Famiglie aggregate per numero di componenti la...,Residenti nelle singole sezioni di censimento ...
3,3,Indirizzario delle sezioni elettorali,Sezioni elettorali
4,4,Nomi di battesimo piu frequenti di sesso masch...,Popolazione residente a Modena dal 1995
5,5,Toponomastica stradale,Numerazione civica del territorio comunale
6,6,Stradario comunale,Bacini scolastici
7,7,Esercizi commerciali,Ristoranti
8,8,Distributori,Strutture ricettive


In [28]:
from tqdm import tqdm   
import time    
import autogen

engine = create_engine('sqlite:///MO_tables.db')
metadata = MetaData()

old = []
tab = uk_tables.drop_duplicates(subset=['query_table', 'candidate_table']).reset_index()

# Start logging with logger_type and the filename to log to
logging_session_id = autogen.runtime_logging.start(logger_type="file", config={"filename": "runtime.log"})
print("Logging session ID: " + str(logging_session_id))

for idx, row in tqdm(tab.iterrows(), total=tab.shape[0]):#uk_tables.shape[0]
    print(f'############ {idx} / {tab.shape[0]} ############')
    table1_name = tab[['query_table']].iloc[idx].values[0]
    table2_name = tab[["candidate_table"]].iloc[idx].values[0]
    table_list = os.listdir('./representative_table_rows_MO')
    #print(idx, table1_name, table2_name)
    with open(f'./representative_table_rows_MO/{table1_name}.txt', 'r') as file:
        table1 = file.read()
        
    with open(f'./representative_table_rows_MO/{table2_name}.txt', 'r') as file:
        table2 = file.read()
        
    s1 = str(CreateTable(Table(table1_name, metadata, autoload_with=engine)).compile(engine))
    s2 = str(CreateTable(Table(table2_name, metadata, autoload_with=engine)).compile(engine))
    for i in range(3):   
        resuts=manager.initiate_chat(
        assistant,
        message=f"""
            These are two joinable tables:
            ##########################
            Table '{table1_name}': {s1} 

            Example:
            {table1}
            ##########################
            Table '{table2_name}': {s2} 

            Example:
            {table2}
            ##########################
            
            make sure to be a lot different from old question:
            {old}
            You are required to GENERATE a natural language question that necessitate joining these two tables. 
            Do not generate the answer.
            Report also the corresponding SQL code block.
            Rembember that table 1 is '{table1_name}' and table 2 name is '{table2_name}'.
            Be sure to use the correct column and table names in the SQL block, and put column names inside ``.
            Generate the natural language question do not add any other info.
            Do not translate columns to english if they are in french, use names as it is.
            Make sure that table names are inside ''.
            If the tables in your opinion do not do join sai TERMINATE.
            WRITE ONLY NATURAL QUESTION:
        """)
        
        
        gt = pd.read_csv("./query_results.csv")
        old = list(gt[(gt['table1'] == table1_name) & (gt['table2'] == table2_name)]['sql'].values)
autogen.runtime_logging.stop()

Logging session ID: e2033e77-6132-4295-ba0f-3df06abb1d54


  0%|                                                                                                | 0/9 [00:00<?, ?it/s]

############ 0 / 9 ############
chat_manager (to Assistant):


            These are two joinable tables:
            ##########################
            Table 'Numero di immigrati stranieri a Modena': 
CREATE TABLE "Numero di immigrati stranieri a Modena" (
	"FID" TEXT, 
	"ANNO" INTEGER, 
	"CITTADINANZA" TEXT, 
	"AREA" TEXT, 
	"PROVENIENZA" TEXT, 
	"SESSO" TEXT, 
	"IMMIGRATI" INTEGER, 
	"OBJECTID" INTEGER, 
	"SHAPE" TEXT
)

 

            Example:
                                                                                       FID  ANNO CITTADINANZA               AREA PROVENIENZA SESSO  IMMIGRATI  OBJECTID                   SHAPE
Numero_di_immigrati_stranieri_a_Modena_preview.fid--6c242f1c_1949384242e_-20f8  2000      ALBANIA Altri paesi Europa    Piemonte     M          4         1 POINT (1652000 4945000)
 Numero_di_immigrati_stranieri_a_Modena_preview.fid--6c242f1c_1949384242e_-710  2016       GRECIA     Unione Europea      Estero     F          4         1 POINT (1652000 4

 11%|█████████▋                                                                             | 1/9 [02:51<22:54, 171.87s/it]

############ 1 / 9 ############
chat_manager (to Assistant):


            These are two joinable tables:
            ##########################
            Table 'Popolazione residente a Modena dal 1995': 
CREATE TABLE "Popolazione residente a Modena dal 1995" (
	"FID" TEXT, 
	"ANNO" INTEGER, 
	"CITTADINANZA" TEXT, 
	"ETA_GRANDI" TEXT, 
	"ETA_QUINQUENNALI" TEXT, 
	"QUARTIERE" TEXT, 
	"ZONA" TEXT, 
	"CENTROPERIFERIA" TEXT, 
	"SESSO" TEXT, 
	"RESIDENTI" INTEGER, 
	"OBJECTID" INTEGER, 
	"SHAPE" TEXT
)

 

            Example:
                                                                                        FID  ANNO CITTADINANZA ETA_GRANDI ETA_QUINQUENNALI                      QUARTIERE                   ZONA  CENTROPERIFERIA   SESSO  RESIDENTI  OBJECTID                   SHAPE
Popolazione_residente_a_Modena_dal_1995_preview.fid--6c242f1c_194938ad992_-4fd0  2017    straniera      45-64            50-54           S.FAUSTINO-MADONNINA        prima periferia Zone periferiche FEMMINE  

...

 33%|█████████████████████████████                                                          | 3/9 [10:44<21:07, 211.30s/it]

############ 3 / 9 ############
chat_manager (to Assistant):


            These are two joinable tables:
            ##########################
            Table 'Indirizzario delle sezioni elettorali': 
CREATE TABLE "Indirizzario delle sezioni elettorali" (
	"FID" TEXT, 
	"SEZIONE_ELETTORALE" INTEGER, 
	"PLESSO" TEXT, 
	"INDIRIZZO_PLESSO" TEXT, 
	"CIVICO_PLESSO" INTEGER, 
	"ESPONENTE_PLESSO" TEXT, 
	"DESCRIZIONE_VIA" TEXT, 
	"CIVICO_PARI_INIZIALE" TEXT, 
	"CIVICO_PARI_FINALE" TEXT, 
	"CIVICO_DISPARI_INIZIALE" TEXT, 
	"CIVICO_DISPARI_FINALE" INTEGER, 
	"OBJECTID" INTEGER, 
	"SHAPE" TEXT
)

 

            Example:
                                                                                     FID  SEZIONE_ELETTORALE             PLESSO              INDIRIZZO_PLESSO  CIVICO_PLESSO ESPONENTE_PLESSO          DESCRIZIONE_VIA CIVICO_PARI_INIZIALE CIVICO_PARI_FINALE CIVICO_DISPARI_INIZIALE  CIVICO_DISPARI_FINALE  OBJECTID                   SHAPE
Indirizzario_delle_sezioni_elettorali_prev

 44%|██████████████████████████████████████▋                                                | 4/9 [13:09<16:27, 197.45s/it]

############ 4 / 9 ############


FileNotFoundError: [Errno 2] No such file or directory: './representative_table_rows_MO/Nomi di battesimo piu frequenti di sesso maschile nel 2021.txt'

In [ ]:
import sqlite3

# Insert data from the DataFrame into the table
#df.to_sql('my_table', con=engine, if_exists='append', index=False)

# Verify the insertion
with sqlite3.connect('MO_tables.db') as connection:
    result = connection.execute("SELECT * FROM 'Quartieri del Comune di Modena'")
    print(result)
    for row in result:
        print(row)

In [ ]:
import os

table1_name = uk_tables[['query_table']].iloc[4491].values[0]
table2_name = uk_tables[["candidate_table"]].iloc[4491].values[0]

table_list = os.listdir('./representative_table_rows')

with open(f'./representative_table_rows/{table1_name}.txt', 'r') as file:
    table1 = file.read()
    
with open(f'./representative_table_rows/{table2_name}.txt', 'r') as file:
    table2 = file.read()

In [ ]:
resuts=manager.initiate_chat(
    assistant,
    message=f"""
        GENERATE a natural language question and the corresponding SQL that necessitate joining these two tables:
        ##########################
        NAME: {table1_name} 
        
        {table1}
        ##########################
        NAME: {table2_name} 
        {table2}
        ##########################
       
        Put the name of the column you use in the question between ''.
        Do not use python code, generate only the question.
        Do not add names of the tables inside the question.
    """,
)

In [ ]:
import zipfile

# Open the ZIP file
with zipfile.ZipFile(DATASET_PATH, "r") as z:
    # List files inside the ZIP
    file_list = [
        f
        for f in z.namelist()
        if not f.startswith("__MACOSX")
        and not f.endswith("/")
        and not f.split("/")[-1].startswith("._")
    ]

    # Extract only filenames
    clean_file_names = [f.split("/")[-1] for f in file_list]

print("Filtered Files:", clean_file_names)

In [ ]:

        
# Create a connection to the SQLite database
conn = sqlite3.connect('CAN_tables.db')
# Create a cursor object
cursor = conn.cursor()

# Define the SQL query
sql_query = """
SELECT T1.`LSOA name`, T1.`Crime type`, T1.`Last outcome category`, COUNT(*) as count 
FROM UK_CSV0000000000004295 T1 
INNER JOIN UK_CSV0000000000010811 T2 
ON T1.`Month` = T2.`Month` 
GROUP BY T1.`LSOA name`, T1.`Crime type`, T1.`Last outcome category`
"""

# Execute the SQL query
cursor.execute(sql_query)

# Fetch all rows from the query result
rows = cursor.fetchall()

# Print the column names
column_names = [description[0] for description in cursor.description]
print(column_names)

# Print the rows
print(rows)

# Close the connection
conn.close()